In [30]:
# !mkdir ~/.kaggle
# !cp '/content/drive/MyDrive/Colab/kaggle_config/kaggle.json' -- ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d demonplus/flower-dataset-102
# !unzip flower-dataset-102.zip

In [95]:
import scipy.io
from google.colab import drive
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
import os
from torch.utils.data import Dataset, DataLoader

In [79]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])
test_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])
img_transforms = {'train': train_tf, 'valid_test': test_tf}

In [92]:
class Flowers102ds(Dataset):
  def __init__(self, imgfolder_path, label_path, setid_path, img_transforms, split='train'):
    self.imgfolder_path = imgfolder_path
    self.labels = scipy.io.loadmat(label_path)['labels'].ravel()
    if split == 'train':
      split = 'trnid'
      self._img_transforms = img_transforms['train']
    elif split == 'valid':
      split = 'valid'
      self._img_transforms = img_transforms['valid_test']
    elif split == 'test':
      split = 'tstid'
      self._img_transforms = img_transforms['valid_test']
    self.setid = scipy.io.loadmat(setid_path)[split].ravel()

  def __len__(self):
    return len(self.setid)

  def __getitem__(self, idx):
    img_id = self.setid[idx]
    label = self.labels[img_id]
    img_path = os.path.join(self.imgfolder_path, f'image_{img_id:05d}.jpg')
    img = Image.open(img_path)
    img_tensor = self._img_transforms(img)
    return img_tensor, label


In [97]:
imgfolder_path = '/content/drive/MyDrive/Colab/datasets/102flowers/jpg/'
label_path = '/content/drive/MyDrive/Colab/datasets/102flowers/imagelabels.mat'
setid_path = '/content/drive/MyDrive/Colab/datasets/102flowers/setid.mat'

In [98]:
train_ds = Flowers102ds(imgfolder_path, label_path, setid_path, img_transforms, split='train')
valid_ds = Flowers102ds(imgfolder_path, label_path, setid_path, img_transforms, split='valid')
test_ds = Flowers102ds(imgfolder_path, label_path, setid_path, img_transforms, split='test')

In [99]:
train_DL = DataLoader(train_ds, batch_size= 16, shuffle=True, drop_last=True)
valid_DL = DataLoader(valid_ds, batch_size=64, shuffle=False)
test_DL = DataLoader(test_ds, batch_size=64, shuffle=False)